# Per-perturbation neural bridge: leave-one-batch-out evaluation

Set `n_validation_batches` below to choose how many distinct batches to evaluate, one at a time. Train a fresh per-perturbation neural bridge and the fitted-linear model for each selected batch, using **every other batch** for training. This gives `n_validation_batches` independent fits per model; batches held out in other runs remain available for training.

Only control inputs are standardised, using that run's training-control statistics. Targets and predictions stay in the dataset's existing expression units. The neural model's inner cross-fitting remains enabled.

Each held-out batch gets two plots of **neural MSE − fitted-linear MSE**: a learning curve against training step, and individual perturbation differences against training-batch count. **Below zero: neural is better. Above zero: neural is worse.**

In [ ]:
import sys
from datetime import datetime
from urllib.parse import quote

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.ticker import MaxNLocator
from omegaconf import OmegaConf

from genetic_perturbation_playground.data.data_loader import DATA_DIR, load_dataset
from genetic_perturbation_playground.model.batch_transfer import (
    BatchTransferModel,
    _bridge_data,
)
from genetic_perturbation_playground.utils.metrics import mse
from genetic_perturbation_playground.utils.plotting import STYLE

sys.path.insert(0, str(DATA_DIR.parent / "scripts"))
from train_batch_transfer_expression import _make_bridge, _split_controls, cross_validate

n_validation_batches = 1  # n separate runs, each holding out one batch

cfg = OmegaConf.merge(
    OmegaConf.load(DATA_DIR.parent / "configs" / "train_batch_transfer_expression.yaml"),
    {
        "name": "perx_leave_one_batch_out",
        "n_validation_batches": n_validation_batches,
        "seed": 0,
        "top_k": 100,
        "bridges": ["neural"],
        "bridge_reg": 0.0001,
        "neural": {
            "per_perturbation": True,
            "weight_decay": 0.0001,
            "n_features_w": 8,
            "width_w": 8,
            "depth": 2,
            "lr": 0.00075,
            "n_steps": 400,
            "cross_fit_folds": 5,
            "device": "cuda" if torch.cuda.is_available() else "cpu",
        },
    },
)
output_dir = DATA_DIR.parent / "outputs" / f"{cfg.name}_{datetime.now():%Y%m%d_%H%M%S}"

## Data and control split

The script selects the validation batches and splits each batch's controls into two halves once. Models use the seen half as inputs; scoring uses the other half as the control baseline. All perturbed cells in the remaining batches enter training.

In [4]:
adata = load_dataset()
n_batches = adata.obs["batch"].nunique()
print(
    f"{cfg.n_validation_batches} separate runs: one held-out batch and {n_batches - 1} training batches each"
)

Loaded /nfs/ghome/live/adhir/genetic_playground/data/replogle_k562_preprocessed.h5ad
  310,385 cells  ×  8,563 genes  |  2,058 perturbations
1 separate runs: one held-out batch and 47 training batches each


## Train and record held-out learning curves

Reuse the script's leave-one-batch-out splits, model factory, and final scoring. A fitting callback uses the neural model's existing checkpoint API to evaluate it **after every training step**, starting at step 1. Each run fits the linear reference once on the same training set.

The curve averages per-perturbation MSE, matching the final score table. It compares both models on the same known perturbations using only the held-out batch's seen controls as inputs. Held-out scores are recorded for inspection; the final checkpoint is used regardless of the curve.

In [ ]:
def fit_with_learning_curve(train, cfg, test, seen):
    """Fit the neural and fitted-linear models; score the neural model after each update.

    The curve is mean per-perturbation MSE on this held-out batch minus the fitted-linear
    model's MSE on the same perturbations. Only training responses enter fitting; held-out
    scores are diagnostics and never select a checkpoint. Use the normal prediction method
    so checkpoint and final losses are both measured in the original target units.
    """
    held_out = np.unique(test.obs["batch"]).item()
    linear = BatchTransferModel(standardise_w=cfg.standardise_w).fit(train)
    bridge = _make_bridge("neural", cfg)
    inputs, _ = _split_controls(test, seen)
    pairs = _bridge_data(inputs)
    labels = pairs.perturbations[pairs.pert]
    known = np.isin(labels, linear.perturbations_)
    labels, truth = labels[known], pairs.y[known]
    linear_mse = mse(linear.predict(labels, inputs), truth).mean()

    def record(step):
        neural_mse = mse(bridge.predict(labels, inputs), truth).mean()
        curve_rows.append(
            {
                "batch": held_out,
                "step": step,
                "neural_mse": neural_mse,
                "linear_mse": linear_mse,
                "mse_difference": neural_mse - linear_mse,
            }
        )

    bridge.fit(train, range(1, cfg.neural.n_steps + 1), record)
    return {"transfer (fit b)": linear, "neural bridge": bridge}

In [ ]:
curve_rows = []
_, per_batch, scores = cross_validate(adata, cfg, fit_models=fit_with_learning_curve)
learning_curves = pd.DataFrame(curve_rows)
losses = scores.query("metric == 'mse'").pivot(
    index=["batch", "perturbation", "n_training_batches"], columns="model", values="value"
)
differences = (
    (losses["neural bridge"] - losses["transfer (fit b)"]).rename("mse_difference").reset_index()
)
output_dir.mkdir(parents=True, exist_ok=True)
scores.to_csv(output_dir / "scores.csv", index=False)
learning_curves.to_csv(output_dir / "learning_curves.csv", index=False)
differences.to_csv(output_dir / "mse_differences.csv", index=False)
print(f"Saved scores and loss differences to {output_dir}")

## Held-out learning curves: neural gain or loss

One curve per held-out batch, with training step on the x-axis. The dashed zero line marks equal MSE. Negative differences favour the neural bridge; positive differences favour the fitted-linear model. The last point equals that batch's mean final per-perturbation MSE difference.

In [ ]:
curve_dir = output_dir / "plots" / "learning_curves"
curve_dir.mkdir(parents=True, exist_ok=True)
with plt.rc_context(STYLE):
    for held_out, table in learning_curves.groupby("batch", sort=False):
        fig, ax = plt.subplots(figsize=(6.4, 3.8), layout="constrained")
        ax.plot(table["step"], table["mse_difference"], color="C0")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.axhline(0, color="black", linestyle="--", linewidth=1)
        ax.set(
            title=f"Held-out batch: {held_out}",
            xlabel="Training step",
            ylabel="Neural MSE − fitted-linear MSE",
        )
        ax.grid(axis="y", alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
        fig.savefig(
            curve_dir / f"batch_{quote(str(held_out), safe='')}.png", dpi=300, bbox_inches="tight"
        )
        plt.show()
        plt.close(fig)

## Final loss difference by training-batch count

Each dot is one perturbation's **neural MSE − fitted-linear MSE** in this held-out batch. The x-axis counts training batches containing that perturbation, excluding the held-out batch. There is no averaging across perturbations or held-out batches in these scatter plots.

In [ ]:
plot_dir = output_dir / "plots" / "by_training_batches"
plot_dir.mkdir(parents=True, exist_ok=True)
with plt.rc_context(STYLE):
    for held_out, table in differences.groupby("batch", sort=False):
        fig, ax = plt.subplots(figsize=(6.4, 3.8), layout="constrained")
        ax.scatter(table["n_training_batches"], table["mse_difference"], s=12, alpha=0.6)
        ax.axhline(0, color="black", linestyle="--", linewidth=1)
        ax.set(
            title=f"Held-out batch: {held_out}",
            xlabel="Training batches containing the perturbation",
            ylabel="Neural MSE − fitted-linear MSE",
            xticks=sorted(table["n_training_batches"].unique()),
        )
        ax.grid(axis="y", alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
        fig.savefig(
            plot_dir / f"batch_{quote(str(held_out), safe='')}.png", dpi=300, bbox_inches="tight"
        )
        plt.show()
        plt.close(fig)

## Mean final gain or loss for each held-out batch

This table averages the individual MSE differences within each batch. Negative means the neural bridge has lower MSE than the fitted-linear model.

In [ ]:
differences.groupby("batch", sort=False)["mse_difference"].mean().to_frame().style.format("{:+.4g}")